# INF201 - Deliverable 3: Battle simulator
**Max Carl Andresen (maand1153) and Sam Mirtaheri (samir9125)**

We continue with the `Character` class from lecture 08 (`game_simulator_v1.py`)
and turn it into a small turn-based battle simulator. Characters can carry a
weapon whose damage is decided by a dice roll, and they can specialise as a
`Warrior` or a `Mage`. A `Battle` object lets two characters fight until one of
them wins.

**GitHub:** https://github.com/madmax502/inf201-h26-deliverables

Select the Python environment in `Deliverable_3/.venv` and run all cells from
the top. The classes are imported from our own modules, which only use the
Python standard library.

## Project structure
| Module | Classes | Task |
|---|---|---|
| `game_simulator_v2.py` | `Character`, `Warrior`, `Mage` | 1, 3 and 4 |
| `dice.py` | `Dice` | 2 |
| `items.py` | `Weapon` | 3 |
| `battle.py` | `Battle` | 5 |

The classes are combined through composition: a `Weapon` *has a* `Dice`, a
`Character` can *have a* `Weapon`, and a `Battle` *has* two characters.
`Warrior` and `Mage` use inheritance: they *are* characters.

## Collaboration
We meet in person and write the code together on one computer. Both group
members contribute to solving the tasks. Because we use one computer and Git
account, the commits are recorded under one person even though the work is shared.

## Setup
We import the classes and fix the random seed, so that every run of the
notebook gives the same dice rolls and the same results.

In [1]:
import inspect
import random
from collections import Counter
from collections.abc import Callable
from contextlib import redirect_stdout
from functools import partial
from io import StringIO

from battle import Battle
from dice import Dice
from game_simulator_v2 import Character, Mage, Warrior
from items import Weapon

# A fixed seed makes the dice rolls, and therefore the results, reproducible
random.seed(201)

To present the classes we use the helper `describe()`. It prints the instance
attributes of an object and every method, property and class constant of its
class, together with the first line of the docstring. It also shows whether a
member is defined in the class itself, overrides a method from the parent
class or is inherited.

In [2]:
def summary_line(member: object) -> str:
    """Return the first paragraph of a docstring as a single line."""
    docstring = inspect.getdoc(member) or ""
    return " ".join(docstring.split("\n\n")[0].split())


def signature_without_self(function: Callable) -> str:
    """Return the signature of a method without the ``self`` parameter."""
    signature = inspect.signature(function)
    parameters = list(signature.parameters.values())[1:]
    return str(signature.replace(parameters=parameters))


def describe(obj: object) -> None:
    """Print the attributes, methods and properties of an object.

    For each member of the class we show whether it is defined in the
    object's own class, overrides a parent class or is inherited.

    Parameters
    ----------
    obj : object
        The object to present.
    """
    cls = type(obj)
    bases = ", ".join(base.__name__ for base in cls.__bases__)
    print(f"class {cls.__name__}({bases})")
    print(f"repr: {obj!r}")

    print("\nInstance attributes:")
    for name, value in vars(obj).items():
        print(f"    {name} = {value!r}")

    # Member names in the order they are defined, from parent to child
    hierarchy = [klass for klass in reversed(cls.__mro__)
                 if klass is not object]
    names = dict.fromkeys(
        name for klass in hierarchy for name in vars(klass)
    )

    print("\nClass constants, methods and properties:")
    for name in names:
        owner = next(klass for klass in cls.__mro__ if name in vars(klass))
        member = vars(owner)[name]
        if isinstance(member, property):
            kind, heading = "property", name
        elif inspect.isfunction(member):
            kind = "method"
            heading = name + signature_without_self(member)
        elif not name.startswith("_"):
            kind, heading = "class constant", f"{name} = {member!r}"
        else:
            continue  # Skip __module__, __dict__, __doc__ and similar

        parents = [klass for klass in cls.__mro__[1:-1]
                   if name in vars(klass)]
        if owner is not cls:
            origin = f"inherited from {owner.__name__}"
        elif parents:
            origin = f"overrides {parents[0].__name__}"
        else:
            origin = f"defined in {cls.__name__}"
        doc = "" if kind == "class constant" else summary_line(member)
        print(f"    {heading}")
        print(f"        [{kind}, {origin}] {doc}")

## Task 1 - Updated `Character` class
In `game_simulator_v1.py`, `attack()` changed the protected attribute `_hp` of
the target directly. In `game_simulator_v2.py` a character only changes its
own HP:

- `take_damage(amount)` lowers the character's own HP, but never below 0.
- `heal(amount)` raises the character's own HP, but never above `hp_max`.
- `attack(target)` finds the damage and calls `target.take_damage()`.
- `hp` and `is_alive` are read-only properties. `is_alive` is `True` while
  `_hp > 0` and `False` otherwise.
- `take_damage()` and `heal()` raise a `ValueError` for negative amounts.
  Otherwise `take_damage(-50)` could heal a character above `hp_max`.

Together this keeps HP between 0 and `hp_max`, and `_hp` is only changed by
the character's own methods.

In [3]:
aria = Character("Aria")
describe(aria)

class Character(object)
repr: <Character name=Aria hp=100 damage=10>

Instance attributes:
    name = 'Aria'
    _hp = 100
    hp_max = 100
    damage = 10
    weapon = None

Class constants, methods and properties:
    __init__(name: str, hp_max: int = 100, damage: int = 10, weapon: items.Weapon | None = None) -> None
        [method, defined in Character] Initialise the Character
    attack(target: 'Character') -> None
        [method, defined in Character] Attack and deal damage to target character
    take_damage(amount: int) -> None
        [method, defined in Character] Reduce own HP by amount, but not below 0
    heal(amount: int) -> None
        [method, defined in Character] Increase own HP by amount, but not above hp_max
    introduce() -> None
        [method, defined in Character] Prints a string that introduces the character
    __repr__() -> str
        [method, defined in Character] Representation of character object
    __str__() -> str
        [method, defined in Chara

In [4]:
goblin = Character("Goblin", hp_max=40, damage=6)
print(goblin)

goblin.take_damage(15)
print(f"After take_damage(15):   {goblin.hp} HP")
goblin.heal(5)
print(f"After heal(5):           {goblin.hp} HP")
goblin.heal(100)
print(f"After heal(100):         {goblin.hp} HP (never above hp_max)")

aria.attack(goblin)  # Calls goblin.take_damage() with Aria's damage
print(f"After Aria's attack:     {goblin.hp} HP")

goblin.take_damage(1000)
print(f"After take_damage(1000): {goblin.hp} HP (never below 0)")
print(f"goblin.is_alive = {goblin.is_alive}")

Character: Goblin with 40 HP and 6 damage
After take_damage(15):   25 HP
After heal(5):           30 HP
After heal(100):         40 HP (never above hp_max)
After Aria's attack:     30 HP
After take_damage(1000): 0 HP (never below 0)
goblin.is_alive = False


In [5]:
# hp and is_alive have no setter, so they cannot be changed from outside
for attribute in ("hp", "is_alive"):
    try:
        setattr(goblin, attribute, 100)
    except AttributeError as error:
        print(f"goblin.{attribute} = 100  ->  AttributeError: {error}")

# Negative amounts are rejected, so HP stays between 0 and hp_max
for method in (goblin.take_damage, goblin.heal):
    try:
        method(-50)
    except ValueError as error:
        print(f"goblin.{method.__name__}(-50)  ->  ValueError: {error}")

goblin.hp = 100  ->  AttributeError: property 'hp' of 'Character' object has no setter
goblin.is_alive = 100  ->  AttributeError: property 'is_alive' of 'Character' object has no setter
goblin.take_damage(-50)  ->  ValueError: amount must be 0 or more, got -50
goblin.heal(-50)  ->  ValueError: amount must be 0 or more, got -50


## Task 2 - The `Dice` class
`Dice(sides=6)` models an n-sided dice. A dice with fewer than two sides
raises a `ValueError`. `roll()` returns `random.randint(1, self.sides)`.
Because the class implements `__call__`, a dice object can also be rolled by
calling it like a function, so `d20()` does the same as `d20.roll()`.
`__repr__` returns strings such as `Dice(sides=20)`.

In [6]:
d6 = Dice()
d20 = Dice(sides=20)
describe(d20)

class Dice(object)
repr: Dice(sides=20)

Instance attributes:
    sides = 20

Class constants, methods and properties:
    __init__(sides: int = 6) -> None
        [method, defined in Dice] Initialise the Dice with a number of sides (default = 6)
    roll() -> int
        [method, defined in Dice] Roll the dice and return a number between 1 and sides
    __call__() -> int
        [method, defined in Dice] Allow the Dice object to be called like a function to roll it
    __repr__() -> str
        [method, defined in Dice] Return a string representation of the Dice object


In [7]:
print(f"Default dice: {d6!r}")
print(f"d20.roll() = {d20.roll()}")
print(f"d20()      = {d20()}  (__call__ also rolls the dice)")

try:
    Dice(sides=1)
except ValueError as error:
    print(f"Dice(sides=1)  ->  ValueError: {error}")

Default dice: Dice(sides=6)
d20.roll() = 3
d20()      = 11  (__call__ also rolls the dice)
Dice(sides=1)  ->  ValueError: A dice must have at least 2 sides


In [8]:
# Roll a d6 6000 times. Each side should come up about 1000 times.
counts = Counter(d6() for _ in range(6000))
for side in range(1, d6.sides + 1):
    print(f"{side}: {counts[side]:>4} {'#' * (counts[side] // 20)}")

1: 1032 ###################################################
2:  982 #################################################
3:  953 ###############################################
4: 1053 ####################################################
5: 1001 ##################################################
6:  979 ################################################


## Task 3 - The `Weapon` class
`Weapon` in `items.py` has a `name` and a `damage_dice`, which is a `Dice`
object. `roll_damage()` rolls this dice. `Character.__init__` now accepts an
optional `weapon` (default `None`). When a weapon is equipped, `attack()` uses
`weapon.roll_damage()` instead of the base damage.

In [9]:
longsword = Weapon("Longsword", Dice(10))
describe(longsword)
print("\nTen damage rolls:", [longsword.roll_damage() for _ in range(10)])

class Weapon(object)
repr: Weapon(name=Longsword, damage_dice=Dice(sides=10))

Instance attributes:
    name = 'Longsword'
    damage_dice = Dice(sides=10)

Class constants, methods and properties:
    __init__(name: str, damage_dice: dice.Dice) -> None
        [method, defined in Weapon] Initialise the Weapon
    roll_damage() -> int
        [method, defined in Weapon] Roll the weapon's damage dice and return the damage
    __repr__() -> str
        [method, defined in Weapon] Representation of weapon object

Ten damage rolls: [6, 10, 2, 10, 4, 10, 6, 9, 5, 7]


The helper `damage_dealt()` lets one character attack another once and returns
how much HP the target lost. We use it to compare attacks without and with a
weapon.

In [10]:
def damage_dealt(attacker: Character, target: Character) -> int:
    """Let ``attacker`` attack ``target`` once and return the damage dealt."""
    hp_before = target.hp
    attacker.attack(target)
    return hp_before - target.hp


# Without a weapon the base damage is used, with a weapon the dice roll
unarmed = Character("Aria")
armed = Character("Aria", weapon=longsword)
dummy = Character("Training dummy", hp_max=1000)
for attacker in (unarmed, armed):
    damages = [damage_dealt(attacker, dummy) for _ in range(10)]
    print(f"weapon={attacker.weapon}: {damages}")

weapon=None: [10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
weapon=Weapon(name=Longsword, damage_dice=Dice(sides=10)): [10, 2, 4, 3, 2, 9, 1, 4, 1, 1]


## Task 4 - Character specialisation
### Warrior
`Warrior` inherits from `Character` and adds `armor` (default 5). It overrides
`take_damage()`: the incoming damage is reduced by the armor, but never below
0, before `Character.take_damage()` is called through `super()`. Since
`attack()` always calls `target.take_damage()`, the armor works no matter who
attacks the warrior.

In [11]:
thora = Warrior("Thora", weapon=Weapon("Battle axe", Dice(12)))
describe(thora)

class Warrior(Character)
repr: <Warrior name=Thora hp=100 damage=10>

Instance attributes:
    name = 'Thora'
    _hp = 100
    hp_max = 100
    damage = 10
    weapon = Weapon(name=Battle axe, damage_dice=Dice(sides=12))
    armor = 5

Class constants, methods and properties:
    __init__(name: str, hp_max: int = 100, damage: int = 10, weapon: items.Weapon | None = None, armor: int = 5) -> None
        [method, overrides Character] Initialise the Warrior
    attack(target: 'Character') -> None
        [method, inherited from Character] Attack and deal damage to target character
    take_damage(amount: int) -> None
        [method, overrides Character] Reduce incoming damage by armor (never below 0), then apply it
    heal(amount: int) -> None
        [method, inherited from Character] Increase own HP by amount, but not above hp_max
    introduce() -> None
        [method, inherited from Character] Prints a string that introduces the character
    __repr__() -> str
        [method, inh

In [12]:
# The same hits against a plain Character and a Warrior with armor=5
for target in (Character("Villager"), Warrior("Thora")):
    for damage in (12, 3):
        target.take_damage(damage)
        print(f"{type(target).__name__:<9} take_damage({damage:>2})  ->  "
              f"{target.hp} HP")

# Armor also works when another character attacks the warrior
rat = Character("Rat", damage=4)
print(f"\nRat (damage=4) attacks Thora: {damage_dealt(rat, thora)} damage")

Character take_damage(12)  ->  88 HP
Character take_damage( 3)  ->  85 HP
Warrior   take_damage(12)  ->  93 HP
Warrior   take_damage( 3)  ->  93 HP

Rat (damage=4) attacks Thora: 0 damage


### Mage
`Mage` adds `mana` and `mana_max` (both 30 by default) and overrides
`attack()`:

- With at least 10 mana (`SPELL_COST`), the mage spends 10 mana on a spell
  that deals double base damage. The spell does not use the weapon.
- With less than 10 mana, the mage makes a normal attack with the equipped
  weapon (or the base damage without a weapon) and restores 5 mana
  (`MANA_RESTORE`), but never above `mana_max`.

In [13]:
eldrin = Mage("Eldrin", weapon=Weapon("Staff", Dice(6)))
describe(eldrin)

class Mage(Character)
repr: <Mage name=Eldrin hp=100 damage=10>

Instance attributes:
    name = 'Eldrin'
    _hp = 100
    hp_max = 100
    damage = 10
    weapon = Weapon(name=Staff, damage_dice=Dice(sides=6))
    mana = 30
    mana_max = 30

Class constants, methods and properties:
    __init__(name: str, hp_max: int = 100, damage: int = 10, weapon: items.Weapon | None = None, mana: int = 30, mana_max: int = 30) -> None
        [method, overrides Character] Initialise the Mage
    attack(target: game_simulator_v2.Character) -> None
        [method, overrides Character] Spell attack (double base damage, no weapon) if mana >= 10, otherwise a normal attack (with weapon if equipped) that restores 5 mana.
    take_damage(amount: int) -> None
        [method, inherited from Character] Reduce own HP by amount, but not below 0
    heal(amount: int) -> None
        [method, inherited from Character] Increase own HP by amount, but not above hp_max
    introduce() -> None
        [method, inhe

Eight attacks in a row show the mana cycle: three spells, then two staff
attacks to get back to 10 mana, then a new spell.

In [14]:
dummy = Character("Training dummy", hp_max=1000)
print("Attack  Type   Mana before  Damage  Mana after")
for number in range(1, 9):
    mana_before = eldrin.mana
    attack_type = "spell" if mana_before >= Mage.SPELL_COST else "staff"
    damage = damage_dealt(eldrin, dummy)
    print(f"{number:>6}  {attack_type:<5}  {mana_before:>11}  {damage:>6}"
          f"  {eldrin.mana:>10}")

Attack  Type   Mana before  Damage  Mana after
     1  spell           30      20          20
     2  spell           20      20          10
     3  spell           10      20           0
     4  staff            0       4           5
     5  staff            5       2          10
     6  spell           10      20           0
     7  staff            0       2           5
     8  staff            5       6          10


## Task 5 - The `Battle` class
`Battle(hero, enemy)` manages turn-based combat between two characters.
`play_round()` lets the hero attack first, and the enemy attacks back only if
it is still alive. `simulate_battle()` plays rounds in a `while` loop until one
of the characters is no longer alive. After each round it prints the HP of
both characters, and at the end it prints the winner.

In [15]:
battle = Battle(Character("Aria"), Character("Goblin", hp_max=40, damage=6))
describe(battle)

class Battle(object)
repr: Battle(hero=<Character name=Aria hp=100 damage=10>, enemy=<Character name=Goblin hp=40 damage=6>)

Instance attributes:
    hero = <Character name=Aria hp=100 damage=10>
    enemy = <Character name=Goblin hp=40 damage=6>

Class constants, methods and properties:
    __init__(hero: game_simulator_v2.Character, enemy: game_simulator_v2.Character) -> None
        [method, defined in Battle] Initialise the Battle
    play_round() -> None
        [method, defined in Battle] Play one round: the hero attacks, then the enemy if still alive
    simulate_battle() -> None
        [method, defined in Battle] Simulate the battle until one character is defeated
    __repr__() -> str
        [method, defined in Battle] Representation of battle object


In [16]:
# One round: the hero attacks first, then the enemy strikes back
battle.play_round()
print("After one round:")
print(f"    {battle.hero}")
print(f"    {battle.enemy}")

# An enemy that dies from the hero's attack does not strike back
battle = Battle(Character("Aria"), Character("Rat", hp_max=5))
battle.play_round()
print("\nAfter one round against a weak enemy:")
print(f"    {battle.hero}")
print(f"    {battle.enemy}, is_alive = {battle.enemy.is_alive}")

After one round:
    Character: Aria with 94 HP and 10 damage
    Character: Goblin with 30 HP and 6 damage

After one round against a weak enemy:
    Character: Aria with 100 HP and 10 damage
    Character: Rat with 0 HP and 10 damage, is_alive = False


### Simulated battles
The helper `run_battle()` prints the two fighters and calls
`simulate_battle()`. Every battle uses new characters with full HP.

In [17]:
def fighter_label(character: Character) -> str:
    """Return a label such as 'Thora (Warrior, Longsword d10)'."""
    weapon = character.weapon
    gear = "no weapon"
    if weapon is not None:
        gear = f"{weapon.name} d{weapon.damage_dice.sides}"
    return f"{character.name} ({type(character).__name__}, {gear})"


def run_battle(hero: Character, enemy: Character) -> None:
    """Print the two fighters and simulate a battle between them."""
    print(f"{fighter_label(hero)}  vs  {fighter_label(enemy)}\n")
    Battle(hero, enemy).simulate_battle()

#### Battle 1: Character vs Character, no weapons
Two identical characters with 100 HP and 10 base damage.

In [18]:
run_battle(Character("Aria"), Character("Bandit"))

Aria (Character, no weapon)  vs  Bandit (Character, no weapon)

Aria HP: 90, Bandit HP: 90
Aria HP: 80, Bandit HP: 80
Aria HP: 70, Bandit HP: 70
Aria HP: 60, Bandit HP: 60
Aria HP: 50, Bandit HP: 50
Aria HP: 40, Bandit HP: 40
Aria HP: 30, Bandit HP: 30
Aria HP: 20, Bandit HP: 20
Aria HP: 10, Bandit HP: 10
Aria HP: 10, Bandit HP: 0
Aria wins!


#### Battle 2: Warrior vs Mage, no weapons

In [19]:
run_battle(Warrior("Thora"), Mage("Eldrin"))

Thora (Warrior, no weapon)  vs  Eldrin (Mage, no weapon)

Thora HP: 85, Eldrin HP: 90
Thora HP: 70, Eldrin HP: 80
Thora HP: 55, Eldrin HP: 70
Thora HP: 50, Eldrin HP: 60
Thora HP: 45, Eldrin HP: 50
Thora HP: 30, Eldrin HP: 40
Thora HP: 25, Eldrin HP: 30
Thora HP: 20, Eldrin HP: 20
Thora HP: 5, Eldrin HP: 10
Thora HP: 5, Eldrin HP: 0
Thora wins!


#### Battle 3: Warrior vs Mage, both with weapons
The warrior has a longsword (d10) and the mage a staff (d6).

In [20]:
run_battle(Warrior("Thora", weapon=Weapon("Longsword", Dice(10))),
           Mage("Eldrin", weapon=Weapon("Staff", Dice(6))))

Thora (Warrior, Longsword d10)  vs  Eldrin (Mage, Staff d6)

Thora HP: 85, Eldrin HP: 90
Thora HP: 70, Eldrin HP: 87
Thora HP: 55, Eldrin HP: 78
Thora HP: 55, Eldrin HP: 77
Thora HP: 55, Eldrin HP: 68
Thora HP: 40, Eldrin HP: 60
Thora HP: 40, Eldrin HP: 59
Thora HP: 39, Eldrin HP: 51
Thora HP: 24, Eldrin HP: 47
Thora HP: 24, Eldrin HP: 38
Thora HP: 24, Eldrin HP: 29
Thora HP: 9, Eldrin HP: 26
Thora HP: 9, Eldrin HP: 19
Thora HP: 9, Eldrin HP: 12
Thora HP: 0, Eldrin HP: 5
Eldrin wins!


#### Battle 4: Character with a weapon vs Warrior without a weapon
The character has a greatsword (d20).

In [21]:
run_battle(Character("Aria", weapon=Weapon("Greatsword", Dice(20))),
           Warrior("Thora"))

Aria (Character, Greatsword d20)  vs  Thora (Warrior, no weapon)

Aria HP: 90, Thora HP: 100
Aria HP: 80, Thora HP: 98
Aria HP: 70, Thora HP: 98
Aria HP: 60, Thora HP: 85
Aria HP: 50, Thora HP: 80
Aria HP: 40, Thora HP: 66
Aria HP: 30, Thora HP: 66
Aria HP: 20, Thora HP: 66
Aria HP: 10, Thora HP: 59
Aria HP: 0, Thora HP: 52
Thora wins!


### Many battles
The result of a single battle with dice depends a lot on luck. We therefore
run 1000 battles for each matchup. Because the hero always attacks first, we
also repeat every matchup with the fighters in swapped roles. The helper
`simulate_quietly()` hides the printed rounds and counts how many rounds were
played.

In [22]:
N_BATTLES = 1000


def simulate_quietly(hero: Character, enemy: Character) -> tuple[bool, int]:
    """Run ``Battle.simulate_battle()`` without printing every round.

    Returns
    -------
    tuple[bool, int]
        Whether the hero won, and the number of rounds played.
    """
    output = StringIO()
    with redirect_stdout(output):
        Battle(hero, enemy).simulate_battle()
    rounds = len(output.getvalue().splitlines()) - 1  # Last line: winner
    return hero.is_alive, rounds


def hero_win_rate(hero_factory: Callable[[], Character],
                  enemy_factory: Callable[[], Character],
                  n_battles: int) -> tuple[float, float]:
    """Simulate many battles, each between two new characters.

    Returns
    -------
    tuple[float, float]
        The share of battles won by the hero, and the average number of
        rounds.
    """
    hero_wins = 0
    total_rounds = 0
    for _ in range(n_battles):
        hero_won, rounds = simulate_quietly(hero_factory(), enemy_factory())
        hero_wins += hero_won
        total_rounds += rounds
    return hero_wins / n_battles, total_rounds / n_battles

In [23]:
longsword = Weapon("Longsword", Dice(10))
greatsword = Weapon("Greatsword", Dice(20))
staff = Weapon("Staff", Dice(6))

# Each value creates a new character with full HP when it is called
fighters = {
    "Character": partial(Character, "Character"),
    "Warrior": partial(Warrior, "Warrior"),
    "Mage": partial(Mage, "Mage"),
    "Character + Greatsword": partial(
        Character, "Character", weapon=greatsword
    ),
    "Warrior + Longsword": partial(Warrior, "Warrior", weapon=longsword),
    "Mage + Staff": partial(Mage, "Mage", weapon=staff),
}
matchups = [
    ("Character", "Character"),
    ("Warrior", "Character"),
    ("Warrior", "Mage"),
    ("Warrior + Longsword", "Mage + Staff"),
    ("Character + Greatsword", "Warrior"),
    ("Character + Greatsword", "Mage + Staff"),
]

print(f"{'':<42}{'A wins':>9}{'A wins':>9}{'Average':>9}")
print(f"{'Fighter A vs fighter B':<42}{'A first':>9}{'B first':>9}"
      f"{'rounds':>9}")
for name_a, name_b in matchups:
    a_first, rounds_1 = hero_win_rate(fighters[name_a], fighters[name_b],
                                      N_BATTLES)
    b_first, rounds_2 = hero_win_rate(fighters[name_b], fighters[name_a],
                                      N_BATTLES)
    average_rounds = (rounds_1 + rounds_2) / 2
    print(f"{name_a + ' vs ' + name_b:<42}{a_first:>9.1%}"
          f"{1 - b_first:>9.1%}{average_rounds:>9.1f}")

                                             A wins   A wins  Average
Fighter A vs fighter B                      A first  B first   rounds
Character vs Character                       100.0%     0.0%     10.0
Warrior vs Character                         100.0%   100.0%     10.0
Warrior vs Mage                              100.0%     0.0%     10.0
Warrior + Longsword vs Mage + Staff            5.4%     1.6%     15.0


Character + Greatsword vs Warrior              0.5%     0.1%     10.0


Character + Greatsword vs Mage + Staff        38.8%    16.8%      8.7


## Discussion of the results
The classes work together as intended. A character's HP only changes through
its own `take_damage()` and `heal()`, and because `attack()` always calls
`target.take_damage()`, the armor of a `Warrior` is used no matter which class
attacks it. Every battle ends with exactly one survivor, because a character
that dies does not strike back.

**The hero has an advantage because it attacks first.** Two identical
characters without weapons (Battle 1) need ten hits each, and the hero wins
every time with 10 HP left. Without weapons all damage is fixed, so these
battles are deterministic. The first attacker also wins every Warrior vs Mage
battle without weapons, and in Battle 2 the warrior won with only 5 HP left.
The advantage remains when the damage is random: the character with a
greatsword won 38.8% of the battles against the mage when it attacked first,
but only 16.8% when the mage attacked first.

**Armor is strong against small hits.** The armor reduces every hit by 5, so a
plain character needs 20 hits instead of 10 to defeat a warrior, and it loses
every time, no matter who starts. Against a d6 staff only a roll of 6 gets
through the armor, and then with only 1 damage. The mage's spells, which deal
20 damage (15 after armor), are what makes the mage dangerous for the warrior.

**A weapon replaces the base damage instead of adding to it.** The base damage
is 10, while a d10 longsword deals 5.5 on average, and only 1.5 on average
against armor. With the dice used here, a weapon can therefore make a
character weaker. The warrior without a weapon beats the mage when it attacks
first, but with a longsword it won only 5.4% of the battles against the mage
with a staff (see also Battle 3). The d20 greatsword deals 10.5 on average,
about the same as the base damage, but only 6 on average against armor. The
character with a greatsword therefore won only 0.5% of its battles against a
warrior.

**Limitation.** `simulate_battle()` has no limit on the number of rounds. If
neither character can deal damage, for example two warriors with armor 5 and
weapons that roll at most 5, the `while` loop never ends. A maximum number of
rounds, or a minimum damage per hit, would prevent this.